# 00 - Pull and Merge Data
Loads the raw input files (RAIS jobs, IBGE census, OD survey, zone crosswalk), builds job_housing_ratio and jobs_accessible_40min, and merges everything into model_df for the analyze notebook.

# Imports

In [54]:
# Libraries needed to load the raw files and build the merged variables
import pandas as pd
import numpy as np
from unidecode import unidecode # text matching across files
from sklearn.preprocessing import StandardScaler  # z-scores predictors before modeling

# Text standardization function

In [55]:
# Takes text value, removes accents (unidecode), converts to uppercase, and strips it

def normalize(text):
    return unidecode(str(text)).upper().strip()

# Load data

In [56]:
rais = pd.read_csv("final_project_data/rais_rmsp.csv") # Job counts by establishment, already filtered to RMSP municipalities

# National census tract data. Brazilian CSVs use latin1 encoding,
ibge_raw = pd.read_csv("final_project_data/Agregados_por_setores_basico_BR.csv", encoding="latin1", sep=";", quotechar='"', decimal=",")

rmsp_codes = rais["id_municipio"].tolist() 
ibge = ibge_raw[ibge_raw["CD_MUN"].isin(rmsp_codes)]

od = pd.read_spss("final_project_data/Banco2023_divulgacao_190225.sav") # 2023 Origin-Destination survey, one row per person or per trip
# This is a Portuguese SPSS format(.sav) file

/var/folders/nb/ydl0f7yd6xggcz7bqtmv3mpr0000gn/T/ipykernel_10085/3455409517.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  rais = pd.read_csv("final_project_data/rais_rmsp.csv") # Job counts by establishment, already filtered to RMSP municipalities
/var/folders/nb/ydl0f7yd6xggcz7bqtmv3mpr0000gn/T/ipykernel_10085/3455409517.py:4: DtypeWarning: Columns (2,3,9,11,13,14,23,25) have mixed types. Specify dtype option on import or set low_memory=False.
  ibge_raw = pd.read_csv("final_project_data/Agregados_por_setores_basico_BR.csv", encoding="latin1", sep=";", quotechar='"', decimal=",")


In [57]:
# Load the crosswalk file that maps each OD survey zone to a district and municipality
zone_crosswalk = pd.read_excel("final_project_data/Corresp2017_2023_190225.xlsx",sheet_name="Divisão Administrativa",header=5,usecols="A:G")
zone_crosswalk = zone_crosswalk.drop([0, 528, 529])
zone_crosswalk.rename(columns={"N°":"zone","Nome":"zone_name", "Nome.2":"dist", "N°.2":"dist_num", "N°.1":"muni_num"}, inplace=True)

In [58]:
# Quick check that each raw file loaded with the expected number of rows/columns
print(f"rais: {rais.shape}")
print(f"ibge: {ibge.shape}")
print(f"od: {od.shape}")
print(f"zone_crosswalk: {zone_crosswalk.shape}")

rais: (1272734, 26)
ibge: (47293, 38)
od: (143038, 147)
zone_crosswalk: (527, 7)


# Creating the jobs_housing_ratio variable
job_housing_ratio is the number of jobs in a zone divided by the number of households in that zone. This variable takes multiple steps to be built because jobs and households are reported at different geographic levels (municipality for RAIS jobs outside the city of Sao Paulo, district for RAIS jobs inside the city, and census tract for IBGE households) and all of them need to be allocated down to the OD survey's zone level using the zone crosswalk file. This is the zone level policy variable that represents job and housing imbalance in the regression

In [59]:
# Lookup table of municipality id to a normalized municipality name
rmsp_munis = ibge[["CD_MUN", "NM_MUN"]].drop_duplicates().rename(columns={"CD_MUN": "id_municipio", "NM_MUN": "name"})
rmsp_munis['muni_name_clean'] = rmsp_munis['name'].apply(normalize)

In [60]:
rais.rename(columns={"quantidade_vinculos_ativos":"jobs"}, inplace=True)

In [61]:
ibge["NM_DIST_clean"] = ibge["NM_DIST"].apply(normalize)
ibge["muni_name_clean"] = ibge["NM_MUN"].apply(normalize)
zone_crosswalk["distrito_clean"] = zone_crosswalk["dist"].apply(normalize)
zone_crosswalk["muni_name_clean"] = zone_crosswalk["Nome.1"].apply(normalize)

/var/folders/nb/ydl0f7yd6xggcz7bqtmv3mpr0000gn/T/ipykernel_10085/1307914793.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ibge["NM_DIST_clean"] = ibge["NM_DIST"].apply(normalize)
/var/folders/nb/ydl0f7yd6xggcz7bqtmv3mpr0000gn/T/ipykernel_10085/1307914793.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ibge["muni_name_clean"] = ibge["NM_MUN"].apply(normalize)


In [62]:
# Before merge: how many rows/nulls zone_crosswalk has going in
print(f"Before merge: {len(zone_crosswalk)} rows")

Before merge: 527 rows


In [63]:
# Attach a municipality id (id_municipio) to the zone crosswalk by matching on the normalized municipality name
zone_crosswalk = zone_crosswalk.merge(rmsp_munis[["id_municipio", "muni_name_clean"]], on="muni_name_clean", how="left")

In [64]:
# After merge: row count should be unchanged, check how many id_municipio came back missing
print(f"After merge: {len(zone_crosswalk)} rows, {zone_crosswalk['id_municipio'].isna().sum()} missing id_municipio")

After merge: 527 rows, 2 missing id_municipio


In [65]:
# Split RAIS jobs into two groups: sp is jobs located inside the city of Sao Paulo (id_municipio 3550308),
# and rmsp is jobs in every other municipality in the metro region
sp = rais[rais["id_municipio"] == 3550308]
rmsp = rais[rais["id_municipio"] != 3550308]

In [66]:
# Sum total jobs inside each RAIS group
jobs_sp_by_district = sp.groupby("distritos_sp")["jobs"].sum().reset_index()
jobs_by_muni = rmsp.groupby("id_municipio")["jobs"].sum().reset_index()

In [67]:
# Before merge: how many rows going into each zone level allocation
print(f"Before merge: {len(zone_crosswalk[zone_crosswalk['muni_name_clean'] == 'SAO PAULO'])} SP zone rows, "
      f"{len(zone_crosswalk[zone_crosswalk['muni_name_clean'] != 'SAO PAULO'])} rest-of-RMSP zone rows")

Before merge: 343 SP zone rows, 184 rest-of-RMSP zone rows


In [68]:
# Allocate jobs down to the OD survey zone level
sp_zones = zone_crosswalk[zone_crosswalk["muni_name_clean"] == 'SAO PAULO'].merge(jobs_sp_by_district, left_on="dist_num", right_on="distritos_sp", how="left")
sp_zones["zones_in_dist"] = sp_zones.groupby("distritos_sp")["zone"].transform("count")
sp_zones["jobs_allocated"] = sp_zones["jobs"] / sp_zones["zones_in_dist"]

rmsp_zones = zone_crosswalk[zone_crosswalk["muni_name_clean"] != 'SAO PAULO'].merge(jobs_by_muni, on="id_municipio", how="left")
rmsp_zones["zones_in_dist"] = rmsp_zones.groupby("dist_num")["zone"].transform("count")
rmsp_zones["jobs_allocated"] = rmsp_zones["jobs"] / rmsp_zones["zones_in_dist"]

In [69]:
# After merge: row counts and how many zones ended up with no jobs figure
print(f"After merge: sp_zones {sp_zones.shape}, {sp_zones['jobs'].isna().sum()} missing jobs")
print(f"After merge: rmsp_zones {rmsp_zones.shape}, {rmsp_zones['jobs'].isna().sum()} missing jobs")

After merge: sp_zones (343, 14), 0 missing jobs
After merge: rmsp_zones (184, 13), 2 missing jobs


In [70]:
# Combine the Sao Paulo zone level job counts and the rest of RMSP zone level job counts into a single table
jobs_by_zone = pd.concat([sp_zones[["zone", "jobs_allocated"]],rmsp_zones[["zone", "jobs_allocated"]]]).dropna(subset=["zone"])

In [71]:
# Same split as above, but for the IBGE census household counts (column v0001)
ibge_sp = ibge[ibge["muni_name_clean"] == 'SAO PAULO']
ibge_rmsp = ibge[ibge["CD_MUN"].astype(str) != "3550308"]

households_by_district = ibge_sp.groupby("NM_DIST_clean")["v0001"].sum().reset_index()
households_by_muni = ibge_rmsp.groupby("CD_MUN")["v0001"].sum().reset_index()

In [72]:
# Before merge: how many rows going into each household zone level allocation
print(f"Before merge: {len(zone_crosswalk[zone_crosswalk['muni_name_clean'] == 'SAO PAULO'])} SP zone rows, "
      f"{len(zone_crosswalk[zone_crosswalk['muni_name_clean'] != 'SAO PAULO'])} rest-of-RMSP zone rows")

Before merge: 343 SP zone rows, 184 rest-of-RMSP zone rows


In [73]:
# Allocate household down to the OD survey zone level
sp_houses_zones = zone_crosswalk[zone_crosswalk["muni_name_clean"] == 'SAO PAULO'].merge(households_by_district, left_on="distrito_clean", right_on="NM_DIST_clean", how="left")
sp_houses_zones["zones_in_dist"] = sp_houses_zones.groupby("distrito_clean")["zone"].transform("count")
sp_houses_zones["households_allocated"] = sp_houses_zones["v0001"] / sp_houses_zones["zones_in_dist"]

rmsp_houses_zones = zone_crosswalk[zone_crosswalk["muni_name_clean"] != 'SAO PAULO'].merge(households_by_muni, left_on="id_municipio", right_on="CD_MUN", how="left")
rmsp_houses_zones["zones_in_muni"] = rmsp_houses_zones.groupby("CD_MUN")["zone"].transform("count")
rmsp_houses_zones["households_allocated"] = rmsp_houses_zones["v0001"] / rmsp_houses_zones["zones_in_muni"]

In [74]:
# After merge: row counts and how many zones ended up with no household figure
print(f"After merge: sp_houses_zones {sp_houses_zones.shape}, {sp_houses_zones['v0001'].isna().sum()} missing v0001")
print(f"After merge: rmsp_houses_zones {rmsp_houses_zones.shape}, {rmsp_houses_zones['v0001'].isna().sum()} missing v0001")

After merge: sp_houses_zones (343, 14), 0 missing v0001
After merge: rmsp_houses_zones (184, 14), 2 missing v0001


In [75]:
# Combine the Sao Paulo zone level household counts and the rest of RMSP zone level household counts into a single table
households_by_zone = pd.concat([sp_houses_zones[["zone", "households_allocated"]],rmsp_houses_zones[["zone", "households_allocated"]]]).dropna(subset=["zone"])

In [76]:
# Before merge: row counts of the two zone level tables being combined
print(f"Before merge: jobs_by_zone {jobs_by_zone.shape}, households_by_zone {households_by_zone.shape}")

Before merge: jobs_by_zone (527, 2), households_by_zone (527, 2)


In [77]:
# Merge jobs and households by zone, then compute job_housing_ratio asjobs divided by households
zone_table = jobs_by_zone.merge(households_by_zone, on="zone", how="outer")
zone_table["job_housing_ratio"] = zone_table["jobs_allocated"] / zone_table["households_allocated"]

In [78]:
# After merge: zone_table row count and how many zones are missing a ratio
print(f"After merge: zone_table {zone_table.shape}, {zone_table['job_housing_ratio'].isna().sum()} missing job_housing_ratio")

After merge: zone_table (527, 4), 2 missing job_housing_ratio


# Creating the jobs_accessible_40min variable
 jobs_accessible_40min represents how many jobs a resident of a given zone can reach by public transit within 40 minutes. Built by looking at every pair of zones in the OD survey, checking whether the average public transit travel time between that pair is 40 minutes or less, and if so, adding the destination zone's job count to the origin zone's accessible job total.

In [79]:
# Count how many (weighted) observations exist for each origin-destinationzone pair, using the trip expansion factor as the weight.
# This weighted count is used below to filter out zone pairs with too few survey responses to trust the average duration.
pair_counts = od.groupby(["zona_o","zona_d"])["Fe_via"].sum()

In [80]:
# Compute the weighted average trip duration for every origin-destination zone pair, using the
# trip expansion factor as the weight. Not filtered to collective transit trips
pair_duration = (od.groupby(["zona_o", "zona_d"]).apply(lambda g: np.average(g["duracao"], weights=g["Fe_via"])))

/var/folders/nb/ydl0f7yd6xggcz7bqtmv3mpr0000gn/T/ipykernel_10085/3434989340.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pair_duration = (od.groupby(["zona_o", "zona_d"]).apply(lambda g: np.average(g["duracao"], weights=g["Fe_via"])))


In [81]:
# Combine the pair level counts and durations into the same table
pair_duration.name = "weighted_duration"
zone_pairs = pd.concat([pair_counts.rename("weighted_n"), pair_duration], axis=1)

In [82]:
# Flag which zone pairs are reachable within 40 minutes. A pair is only marked reachable (1) or not reachable (0) if it
# has at least MIN_OBS weighted observations; otherwise it is left as missing (nan) because the average duration
# is not considered reliable with too few responses
zone_pairs = pd.concat([pair_counts.rename("weighted_n"), pair_duration.rename("weighted_duration")],axis=1)
zp = zone_pairs.reset_index()

MIN_OBS = 5

zp["reachable"] = np.where(zp["weighted_n"] >= MIN_OBS,(zp["weighted_duration"] <= 40).astype(int),np.nan)

In [83]:
# Before merge: how many zone pairs are being matched to a destination job count
print(f"Before merge: {len(zp)} zone pairs")

Before merge: 33192 zone pairs


In [84]:
# Attach the destination zone's allocated job count to each zone pair
zp = zp.merge(zone_table[["zone", "jobs_allocated"]],left_on="zona_d",right_on="zone",how="left")

In [85]:
# After merge: row count should be unchanged, check how many pairs got no job count
print(f"After merge: {len(zp)} zone pairs, {zp['jobs_allocated'].isna().sum()} missing jobs_allocated")

After merge: 33192 zone pairs, 14 missing jobs_allocated


In [86]:
# Sum up the job counts of every destination zone that is reachable within 40 minutes for each origin zone
jobs_accessible_40min = (zp[zp["reachable"] == 1].groupby("zona_o")["jobs_allocated"].sum().reset_index()
    .rename(columns={"zona_o": "zona", "jobs_allocated": "jobs_accessible_40min"})
)

In [87]:
# Before merge: how many origin zones have an accessible jobs total so far
print(f"Before merge: {len(jobs_accessible_40min)} origin zones")

Before merge: 514 origin zones


In [88]:
# Attach household counts for each origin zone
jobs_accessible_40min = jobs_accessible_40min.merge(households_by_zone.rename(columns={"zone": "zona"}),on="zona",how="left")

In [89]:
# After merge: row count should be unchanged, check how many origin zones got no household count
print(f"After merge: {len(jobs_accessible_40min)} origin zones, {jobs_accessible_40min['households_allocated'].isna().sum()} missing households_allocated")

After merge: 514 origin zones, 1 missing households_allocated


In [90]:
# Divide accessible jobs by households in the origin zone
jobs_accessible_40min["jobs_accessible_per_household"] = (jobs_accessible_40min["jobs_accessible_40min"] / jobs_accessible_40min["households_allocated"])

# Building model_df
model_df starts as the OD survey (one row per trip or person) and gets the two zone level variables, job_housing_ratio and jobs_accessible_per_household, merged onto it by matching each respondent's home zone.

In [91]:
# Before merge: how many rows the trip level OD survey has going in
print(f"Before merge: od {od.shape}")

Before merge: od (143038, 147)


In [92]:
# Merge the trip level OD survey with the two zone level variables built earlier, matching on the respondent's home zone (zona)
model_df = od.merge(zone_table[["zone", "job_housing_ratio"]], left_on="zona", right_on="zone", how="left"
    ).merge(jobs_accessible_40min[["zona", "jobs_accessible_per_household"]],on="zona", how="left")

In [93]:
# After merge: row count should be unchanged, check how many trips got no zone level variables
print(f"After merge: model_df {model_df.shape}, "
      f"{model_df['job_housing_ratio'].isna().sum()} missing job_housing_ratio, "
      f"{model_df['jobs_accessible_per_household'].isna().sum()} missing jobs_accessible_per_household")

After merge: model_df (143038, 150), 273 missing job_housing_ratio, 273 missing jobs_accessible_per_household


modoprin is mapped down to 8 broader, policy relevant groups: Walk/Bike, Car (driver), Car (passenger), Moto, Rail transit, Bus, School/chartered, Taxi/app, and Other.

In [94]:
# Dictionary mapping each detailed transport mode label in the OD survey to a broader category used in the regression.
mode_map = {
    "A pé": "Walk/Bike",
    "Bicicleta": "Walk/Bike",
    "Dirigindo automóvel": "Car (driver)",
    "Passageiro de automóvel": "Car (passenger)",
    "Dirigindo moto": "Moto",
    "Passageiro de moto": "Moto",
    "Passageiro de mototáxi": "Moto",
    "Metrô": "Rail transit",
    "Trem": "Rail transit",
    "Monotrilho": "Rail transit",
    "Ônibus/micro-ônibus/van do município de São Paulo": "Bus",
    "Ônibus/micro-ônibus/van metropolitano": "Bus",
    "Ônibus/micro-ônibus/van de outros municípios": "Bus",
    "Transporte escolar": "School/chartered",
    "Transporte fretado": "School/chartered",
    "Táxi convencional": "Taxi/app",
    "Táxi não convencional / aplicativo": "Taxi/app",
    "Outros": "Other",
}

In [95]:
# Apply the mapping to create the new mode_group column
model_df["mode_group"] = model_df["modoprin"].map(mode_map).astype("category")

# Creating the income_usd variable
income_usd converts renda_fa to US$, using the 2023 average USD/BRL exchange rate. This is a trade-off variable.

In [99]:
# 2023 average USD/BRL exchange rate (R$4.99 = US$1), source: exchange-rates.org 2023 history
USD_BRL_2023_AVG = 4.85

model_df["income_usd"] = model_df["renda_fa"] / USD_BRL_2023_AVG


num_vars holds the continuous predictors that will be z-scored. cat_vars holds the categorical predictors that will get a reference category set. outcome is the dependent variable, trip duration in minutes. group is the column used as the random intercept grouping variable in the mixed effects model, the respondent's home zone.

In [100]:
num_vars = ["job_housing_ratio", "jobs_accessible_per_household", "qt_auto", "income_usd", "idade"]
cat_vars = ["sexo", "motivo", "mode_group"]
outcome = "duracao"
group = "zona"


In [101]:
model_df = model_df.dropna(subset=["duracao"]) # Drop rows with no recorded trip duration

Each continuous predictor is converted to a z-score. This makes the resulting regression coefficients comparable to each other on the forest plot.

In [45]:
# Standardize each continuous predictor and store the results in new columns with a _z suffix
scaler = StandardScaler()
model_df[[f"{v}_z" for v in num_vars]] = scaler.fit_transform(model_df[num_vars])

Since one category has to be chosen as the baseline that every other category is compared against, sexo uses Masculino as the reference, motivo uses whichever trip purpose is most common in the data, and mode_group uses Car (driver) as the reference.

In [46]:
# Cast to category type, then reorder so "Masculino" is first
model_df["sexo"] = model_df["sexo"].astype("category")
model_df["sexo"] = model_df["sexo"].cat.reorder_categories(
    sorted(model_df["sexo"].cat.categories, key=lambda x: x != "Masculino")
)

In [47]:
# Cast to category type, then reorder so the most frequent purpose is first
model_df["motivo"] = model_df["motivo"].astype("category")
top_motivo = model_df["motivo"].value_counts().idxmax()
model_df["motivo"] = model_df["motivo"].cat.reorder_categories(
    [top_motivo] + [c for c in model_df["motivo"].cat.categories if c != top_motivo]
)

In [48]:
# Reorder so "Car (driver)" is first and becomes the reference category
model_df["mode_group"] = model_df["mode_group"].cat.reorder_categories(
    ["Car (driver)"] + [c for c in model_df["mode_group"].cat.categories if c != "Car (driver)"]
)

# Save merged data

In [ ]:
# Save model_df so the analyze notebook can load it directly, already cleaned and feature engineered
model_df.to_pickle("final_project_data/interim/model_df.pkl")
print(f"Saved model_df: {model_df.shape}")